# Navigating Text in High Dimensions

*Dataharvest 2026 — hands-on notebook*

Keyword search is great when you know what you're looking for. But what about the
structure you *don't* know is there — the topics, the patterns, the outliers?

In this notebook we'll take a pile of tweets and turn it into a **map you can explore**:

1. Turn text into numbers (**embeddings**)
2. Measure **meaning** with cosine similarity → **semantic search**
3. **Project** to 2D with UMAP and plot it
4. **Find groups** with HDBSCAN — and see what *doesn't* fit

You don't need any machine-learning background. Just run each cell with **Shift+Enter**.
A few cells are marked **TRY IT** — change a value and re-run to see what happens.

Slides: <https://textembeddings.resolve.works>


## 0. Setup

This installs the three libraries we need. On Colab it takes about a minute.
Everything else (pandas, scikit-learn) is already there.

In [ ]:
!pip install -q sentence-transformers umap-learn plotly

In [ ]:
import textwrap
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

pd.set_option("display.max_colwidth", 100)

## 1. Embedding text

An **embedding model** reads a piece of text and turns it into a list of numbers — a
**vector**. We'll use [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2):
small, fast, and runs fine on a laptop.

The first run downloads the model (~80 MB).

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embedding = model.encode("The sun is shining")

print("type:  ", type(embedding).__name__)
print("shape: ", embedding.shape, "  <- 384 numbers")
print("first 8 values:")
print(embedding[:8])

Every piece of text — a word, a sentence, a tweet — becomes **384 numbers**.
That's it. The text now lives at a point in a 384-dimensional space.

## 2. Comparing meaning

Texts with similar meaning get similar vectors. We compare them with **cosine similarity**:
1.0 means "same direction / same meaning", 0.0 means "unrelated".

Let's reproduce the example from the slides.

In [ ]:
query = "The sun is shining"

candidates = [
    "Bring an umbrella",
    "The weather is great",
    "It's daytime",
]

q_emb = model.encode(query)
c_emb = model.encode(candidates)
scores = util.cos_sim(q_emb, c_emb)[0]

for text, score in sorted(zip(candidates, scores), key=lambda x: -x[1]):
    print(f"{score:.3f}   {text}")

Notice the model has **no keywords in common** to go on — "daytime" scores highest
because it's *closest in meaning*, not because it shares words.

> **TRY IT** — change the `query` and `candidates` above to your own sentences and re-run.

## 3. Load some real data

Now let's work with a real corpus: tweets from the
[TweetTopic](https://huggingface.co/datasets/cardiffnlp/tweet_topic_single) dataset,
where humans labelled each tweet with one of **6 topics**. We'll keep those labels
aside as a "ground truth" to check our results against later.

We fetch it live and tidy up the placeholders the dataset uses for URLs and usernames.

In [ ]:
import io, re, urllib.request

URL = ("https://huggingface.co/datasets/cardiffnlp/tweet_topic_single/"
       "resolve/main/dataset/split_temporal/train_2020.single.json")

req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
raw = urllib.request.urlopen(req).read().decode("utf-8")
df = pd.read_json(io.StringIO(raw), lines=True)

def clean(text):
    text = text.replace("{{URL}}", "").replace("{{USERNAME}}", "@user")
    text = re.sub(r"\{@(.*?)@\}", r"@\1", text)   # {@Some Name@} -> @Some Name
    return re.sub(r"\s+", " ", text).strip()

df["text"] = df["text"].map(clean)
print(f"{len(df)} tweets loaded")
df["label_name"].value_counts()

The topics are very unbalanced (lots of sports and pop culture, little arts).
For a clean demo we take an **even sample** — up to 100 tweets per topic — and shuffle them.

In [ ]:
N_PER_TOPIC = 100

parts = []
for topic, group in df.groupby("label_name"):
    parts.append(group.sample(min(len(group), N_PER_TOPIC), random_state=42))

df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"{len(df)} tweets after sampling")
print(df["label_name"].value_counts())
df[["text", "label_name"]].head()

## 4. Embed the whole corpus

One call embeds every tweet. ~550 short texts take a few seconds on a laptop.

In [ ]:
texts = df["text"].tolist()
embeddings = model.encode(texts, show_progress_bar=True)

print("embeddings shape:", embeddings.shape, "  <- (tweets, 384)")

## 5. Semantic search

Now we can search by **meaning** instead of keywords: embed a query, then find the
tweets whose vectors point in the most similar direction.

In [ ]:
def search(query, k=5):
    q_emb = model.encode(query)
    scores = util.cos_sim(q_emb, embeddings)[0]
    top = scores.argsort(descending=True)[:k]
    for i in top:
        i = int(i)
        print(f"{scores[i]:.3f}   {df['text'].iloc[i]}")

search("a thrilling football match")

> **TRY IT** — change the query. Try something with no obvious keywords, e.g.
`"new gadget release"`, `"stock market"`, or `"feeling stressed"`.

## 6. Project to 2D with UMAP

Each tweet is a point in **384 dimensions** — impossible to picture. **UMAP** squeezes
that down to **2 dimensions** while keeping neighbours close to neighbours, so we can
plot it.

We colour each point by its **human topic label**. Hover over a point to read the tweet.

In [ ]:
from umap import UMAP

reducer = UMAP(
    n_neighbors=15,     # how much neighbourhood to consider (local <-> global)
    min_dist=0.1,       # how tightly points may pack together
    n_components=2,
    metric="cosine",
    random_state=42,
)
coords = reducer.fit_transform(embeddings)
df["x"], df["y"] = coords[:, 0], coords[:, 1]
df["hover"] = df["text"].map(lambda t: "<br>".join(textwrap.wrap(t, 60)))
print("done — projected to", coords.shape[1], "dimensions")

In [ ]:
import plotly.express as px

fig = px.scatter(
    df, x="x", y="y", color="label_name",
    custom_data=["hover"], title="Tweets in 2D, coloured by human topic",
    width=900, height=600, opacity=0.8,
)
fig.update_traces(hovertemplate="%{customdata[0]}<extra></extra>")
fig.update_layout(legend_title_text="topic", xaxis_title="", yaxis_title="")
fig.show()

Tweets about the same topic land near each other — **without anyone telling UMAP
about the labels**. It only saw the 384-number vectors.

> **TRY IT** — set `n_neighbors` to `5` (very local) or `50` (more global) above and
re-run both cells. Watch the shape of the map change.

## 7. Find groups automatically with HDBSCAN

What if we *didn't* have the human labels? **HDBSCAN** finds dense groups on its own —
and labels points in sparse areas as **noise** (`-1`), i.e. "doesn't fit any group".

We run it on the 2D coordinates so the groups match what we see on the plot.

In [ ]:
from sklearn.cluster import HDBSCAN

clusterer = HDBSCAN(min_cluster_size=15)
df["cluster"] = clusterer.fit_predict(coords)

n_clusters = df.loc[df["cluster"] != -1, "cluster"].nunique()
n_noise = int((df["cluster"] == -1).sum())
print(f"found {n_clusters} clusters")
print(f"{n_noise} tweets left as noise (don't fit any group)")

In [ ]:
df["group"] = df["cluster"].map(lambda c: "noise" if c == -1 else f"cluster {c}")

order = ["noise"] + [f"cluster {c}" for c in sorted(df.loc[df['cluster'] != -1, 'cluster'].unique())]
cmap = {"noise": "lightgray"}

fig = px.scatter(
    df, x="x", y="y", color="group",
    custom_data=["hover"], category_orders={"group": order},
    color_discrete_map=cmap, title="The same map, coloured by discovered cluster",
    width=900, height=600, opacity=0.8,
)
fig.update_traces(hovertemplate="%{customdata[0]}<extra></extra>")
fig.update_layout(legend_title_text="", xaxis_title="", yaxis_title="")
fig.show()

> **TRY IT** — change `min_cluster_size` (e.g. `8` for more, smaller clusters, or
`30` for fewer, bigger ones) and re-run both cells.

## 8. What is each cluster about?

A cluster is just a bag of tweets. To understand it, we pull out the **words that are
distinctive** to each cluster (using TF-IDF), plus a few example tweets.

This is the journalist's payoff: *what are people talking about?*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

clusters = sorted(df.loc[df["cluster"] != -1, "cluster"].unique())
cluster_docs = [" ".join(df.loc[df["cluster"] == c, "text"]) for c in clusters]

stop = list(ENGLISH_STOP_WORDS) + ["user", "amp", "rt", "http", "https", "com"]
vec = TfidfVectorizer(stop_words=stop, token_pattern=r"[A-Za-z]{3,}", max_features=3000)
tfidf = vec.fit_transform(cluster_docs)
terms = vec.get_feature_names_out()

for row, c in zip(tfidf, clusters):
    scores = row.toarray().ravel()
    keywords = [terms[j] for j in scores.argsort()[::-1][:8]]
    members = df.loc[df["cluster"] == c]
    print(f"CLUSTER {c}  ({len(members)} tweets)")
    print("  keywords:", ", ".join(keywords))
    for t in members["text"].head(3):
        print("   •", t[:110])
    print()

How well do the discovered clusters line up with the **human topics**? This table
cross-tabulates them — each row is a cluster, each column a human label.

In [ ]:
pd.crosstab(df["group"], df["label_name"])

## 9. The outliers

The tweets HDBSCAN left as **noise** are the ones that didn't fit any group — often the
most interesting for an investigation. Here are a few.

In [ ]:
for t in df.loc[df["cluster"] == -1, "text"].head(10):
    print("•", t)

## That's the workflow

From a pile of text to an explorable map, with clusters and outliers — and you never
had to tell the machine what to look for.

**Use it on your own data:** replace the data in step 3 with any column of text
(a CSV of articles, comments, FOI responses, transcripts...) and run the rest.

**Tools to keep exploring:**
- [sbert.net](https://sbert.net) — the embeddings library
- [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — compare embedding models
- [UMAP docs](https://umap-learn.readthedocs.io/) · [TensorFlow Projector](https://projector.tensorflow.org/)
- Turn documents into text: [docling](https://github.com/docling-project/docling), [markitdown](https://github.com/microsoft/markitdown)
